In [66]:
import pandas as pd
from sqlalchemy import create_engine

# Connect
engine = create_engine(
    "mssql+pyodbc://.\\SQLEXPRESS/PortfolioProject_MarketingAnalytics"
    "?driver=SQL+Server&trusted_connection=yes"
)

# Find ALL real table names
df_tables = pd.read_sql("""
    SELECT TABLE_SCHEMA, TABLE_NAME 
    FROM INFORMATION_SCHEMA.TABLES 
    WHERE TABLE_TYPE = 'BASE TABLE'
""", engine)

print(df_tables)

  TABLE_SCHEMA        TABLE_NAME
0          dbo         customers
1          dbo         geography
2          dbo          products
3          dbo  customer_journey
4          dbo  customer_reviews
5          dbo   engagement_data


C:\Users\hp\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1110.1'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


In [67]:
# ── LOAD ALL 6 TABLES ─────────────────────────────
df_customers        = pd.read_sql("SELECT * FROM customers", engine)
df_geography        = pd.read_sql("SELECT * FROM geography", engine)
df_products         = pd.read_sql("SELECT * FROM products", engine)
df_customer_journey = pd.read_sql("SELECT * FROM customer_journey", engine)
df_customer_reviews = pd.read_sql("SELECT * FROM customer_reviews", engine)
df_engagement_data  = pd.read_sql("SELECT * FROM engagement_data", engine)

print("✅ All 6 tables loaded!")

✅ All 6 tables loaded!


In [68]:
df_customer_reviews.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1363 entries, 0 to 1362
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ReviewID    1363 non-null   int64 
 1   CustomerID  1363 non-null   int64 
 2   ProductID   1363 non-null   int64 
 3   ReviewDate  1363 non-null   object
 4   Rating      1363 non-null   int64 
 5   ReviewText  1363 non-null   object
dtypes: int64(4), object(2)
memory usage: 64.0+ KB


In [69]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [70]:
import pyodbc
import nltk

In [71]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [72]:
query = "SELECT ReviewID, CustomerID, ProductID, ReviewDate, Rating, ReviewText FROM fact_customer_reviews"


In [73]:
sia = SentimentIntensityAnalyzer()

In [74]:
def calculate_sentiment(review):
    # Get the sentiment scores for the review text
    sentiment = sia.polarity_scores(review)
    # Return the compound score, which is a normalized score between -1 (most negative) and 1 (most positive)
    return sentiment['compound']

In [75]:
# Define a function to categorize sentiment using both the sentiment score and the review rating
def categorize_sentiment(score, rating):
    # Use both the text sentiment score and the numerical rating to determine sentiment category
    if score > 0.05:  # Positive sentiment score
        if rating >= 4:
            return 'Positive'  # High rating and positive sentiment
        elif rating == 3:
            return 'Mixed Positive'  # Neutral rating but positive sentiment
        else:
            return 'Mixed Negative'  # Low rating but positive sentiment
    elif score < -0.05:  # Negative sentiment score
        if rating <= 2:
            return 'Negative'  # Low rating and negative sentiment
        elif rating == 3:
            return 'Mixed Negative'  # Neutral rating but negative sentiment
        else:
            return 'Mixed Positive'  # High rating but negative sentiment
    else:  # Neutral sentiment score
        if rating >= 4:
            return 'Positive'  # High rating with neutral sentiment
        elif rating <= 2:
            return 'Negative'  # Low rating with neutral sentiment
        else:
            return 'Neutral'  # Neutral rating and neutral sentiment

In [59]:
# Check if it's even a DataFrame
print(type(customer_reviews_df))
print(customer_reviews_df)

<class 'tuple'>
()


In [61]:
print(type(df_customer_reviews))  # always confirm it's a DataFrame before using it

<class 'pandas.core.frame.DataFrame'>


In [63]:
def sentiment_bucket(score):
    if score >= 0.5:
        return '0.5 to 1.0'  # Strongly positive sentiment
    elif 0.0 <= score < 0.5:
        return '0.0 to 0.49'  # Mildly positive sentiment
    elif -0.5 <= score < 0.0:
        return '-0.49 to 0.0'  # Mildly negative sentiment
    else:
        return '-1.0 to -0.5'  # Strongly negative sentiment

In [76]:
# Apply sentiment analysis to calculate sentiment scores for each review
df_customer_reviews['SentimentScore']    = df_customer_reviews['ReviewText'].apply(calculate_sentiment)

# Apply sentiment categorization using both text and rating
df_customer_reviews['SentimentCategory'] = df_customer_reviews.apply(
    lambda row: categorize_sentiment(row['SentimentScore'], row['Rating']), axis=1)
# Apply sentiment bucketing to categorize scores into defined ranges

df_customer_reviews['SentimentBucket']   = df_customer_reviews['SentimentScore'].apply(sentiment_bucket)

# ── PREVIEW & SAVE ────────────────────────────────
display(df_customer_reviews[['ReviewText','Rating','SentimentScore','SentimentCategory','SentimentBucket']].head())

# Save the DataFrame with sentiment scores, categories, and buckets to a new CSV file
df_customer_reviews.to_csv('fact_customer_reviews_with_sentiment.csv', index=False)
print("✅ Saved to CSV!")

,ReviewText,Rating,SentimentScore,SentimentCategory,SentimentBucket
0,"Average experience, nothing special.",3,-0.3089,Mixed Negative,-0.49 to 0.0
1,The quality is top-notch.,5,0.0000,Positive,0.0 to 0.49
2,Five stars for the quick delivery.,4,0.0000,Positive,0.0 to 0.49
3,"Good quality, but could be cheaper.",3,0.2382,Mixed Positive,0.0 to 0.49
4,"Average experience, nothing special.",3,-0.3089,Mixed Negative,-0.49 to 0.0


✅ Saved to CSV!
